# Construcción del arrays NumPy de 8 capas por sensor + RobustScaler

Pasos:
1. Leer los GeoTIFF de 4 bandas (`NDVI, TC_Brightness, TC_Greenness, TC_Wetness`).
2. Construir el array `(4 bandas, alto, ancho)` por temporada.
3. Combinar las temporadas de seca y húmeda en un solo array, esto ya sería las`(8 bandas, alto, ancho)` por sensor.
4. Aplicar el método `RobustScaler` (no acepta valores NaN), entonces se filtraron los pixeles para que no falle o contamine la mediana/IQR.Para eso se usa el método `mask_valida.`
5. Guardar el array escalado (`.npy`) y el orden de capas (`.json`) por sensor.



In [10]:
# Ruta de la carpeta de Drive donde estan los .tif exportados
RUTA_TIF = '/content/drive/MyDrive/ANP/Landsat/SVerde_Landsat/SVD_tif-2000'

PREFIJO    = 'SVD'
AÑO        = 2000
SENSORES   = ['L5', 'L7']
TEMPORADAS = ['seca', 'humeda']
NOMBRES_BANDAS = ['NDVI', 'TC_Brightness', 'TC_Greenness', 'TC_Wetness']

# Carpeta local donde se guardaran los .npy y .json de salida
RUTA_SALIDA ='/content/drive/MyDrive/ANP/Landsat/SVerde_Landsat/SVD_tif-2000/Nympy2000'

In [11]:
import os
import glob
import json
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
def buscar_tif(patron, avisar_si_hay_varios=True):     # si hay mas de uno agarramos el mas nuevo, pero igual avisamos
    candidatos = glob.glob(patron)
    if len(candidatos) == 0:
        raise FileNotFoundError(f"No se encontro tif con patron: {patron}")

    if len(candidatos) > 1:
        candidatos.sort(key=os.path.getmtime, reverse=True)
        if avisar_si_hay_varios:
            print(f"  Aviso: {len(candidatos)} archivos coinciden, usando el mas reciente: {os.path.basename(candidatos[0])}")

    return candidatos[0]


def encontrar_tif(prefijo, año, sensor, temporada, carpeta):
    patron = os.path.join(carpeta, f"{prefijo}_{año}_{sensor}_{temporada}*.tif")
    candidatos = [f for f in glob.glob(patron) if 'bandas_corr' not in f]      #descarto los de bandas_corr
    if not candidatos:
        raise FileNotFoundError(f"No se encontro tif para {sensor} {temporada} con patron: {patron}")
    if len(candidatos) > 1:
        candidatos.sort(key=os.path.getmtime, reverse=True)
        print(f"  Aviso: {len(candidatos)} archivos coinciden para {sensor}-{temporada}, usando el mas reciente")
    return candidatos[0]


def leer_tif_a_array(path):       # devuelve (bandas, H, W) en float32 + el profile por si hace falta despues
    with rasterio.open(path) as src:
        arr = src.read().astype('float32')
        if src.nodata is not None and not np.isnan(src.nodata):
            arr[arr == src.nodata] = np.nan
        perfil = src.profile
    return arr, perfil


def construir_array_8capas(sensor, prefijo, año, carpeta, temporadas=TEMPORADAS, nombres_bandas=NOMBRES_BANDAS):
    partes = []
    orden_capas = []
    perfil_ref = None

    for temporada in temporadas:
        path = encontrar_tif(prefijo, año, sensor, temporada, carpeta)
        arr, perfil = leer_tif_a_array(path)

        if perfil_ref is None:
            perfil_ref = perfil
        else:  # las dos temporadas tienen que tener el mismo shape, sino algo se exporto mal en Earth Engine
            dim_ref = (perfil_ref['height'], perfil_ref['width'])
            if arr.shape[1:] != dim_ref:
                raise ValueError(f"Dimensiones de {sensor}-{temporada} {arr.shape[1:]} no coinciden con {dim_ref}")

        partes.append(arr)
        for b in nombres_bandas:
            orden_capas.append(f"{b}_{temporada}")

    array_8 = np.concatenate(partes, axis=0)
    return array_8, orden_capas, perfil_ref


def aplicar_robust_scaler(array_8capas):
    n_bandas, H, W = array_8capas.shape
    X = array_8capas.reshape(n_bandas, H * W).T          # (n_pixeles, 8)
    mask_valida = ~np.isnan(X).any(axis=1)

    if mask_valida.sum() == 0:
        raise ValueError("Todos los pixeles son NaN; no hay datos validos para escalar.")

    scaler = RobustScaler()
    X_escalado = np.full_like(X, np.nan)
    X_escalado[mask_valida] = scaler.fit_transform(X[mask_valida])

    array_escalado = X_escalado.T.reshape(n_bandas, H, W)
    return array_escalado, scaler

## Ejecutar para cada sensor

In [13]:
os.makedirs(RUTA_SALIDA, exist_ok=True)

resultados = {}

for sensor in SENSORES:
    print(f"\nProcesando sensor {sensor}...")
    array_8, orden_capas, perfil = construir_array_8capas(sensor, PREFIJO, AÑO, RUTA_TIF)
    print(f"  Array combinado (seca+humeda): shape = {array_8.shape}")

    array_escalado, scaler = aplicar_robust_scaler(array_8)
    print(f"  Array escalado: shape = {array_escalado.shape}, "
          f"NaN preservados = {np.isnan(array_escalado).sum()}")

    resultados[sensor] = {
        'array': array_escalado,
        'orden_capas': orden_capas,
        'perfil': perfil,
        'scaler': scaler,
    }

    ruta_npy  = os.path.join(RUTA_SALIDA, f'{PREFIJO}_{AÑO}_{sensor}_8capas_escalado.npy')
    ruta_json = os.path.join(RUTA_SALIDA, f'{PREFIJO}_{AÑO}_{sensor}_8capas_orden.json')

    np.save(ruta_npy, array_escalado)
    with open(ruta_json, 'w') as f:
        json.dump(orden_capas, f, ensure_ascii=False, indent=2)

    print(f"  Guardado: {ruta_npy}")
    print(f"  Guardado: {ruta_json}")

print("\nOrden de capas (igual para todos los sensores):")
print(resultados[SENSORES[0]]['orden_capas'])



Procesando sensor L5...
  Aviso: 2 archivos coinciden para L5-humeda, usando el mas reciente
  Array combinado (seca+humeda): shape = (8, 763, 542)
  Array escalado: shape = (8, 763, 542), NaN preservados = 342776
  Guardado: /content/drive/MyDrive/ANP/Landsat/SVerde_Landsat/SVD_tif-2000/Nympy2000/SVD_2000_L5_8capas_escalado.npy
  Guardado: /content/drive/MyDrive/ANP/Landsat/SVerde_Landsat/SVD_tif-2000/Nympy2000/SVD_2000_L5_8capas_orden.json

Procesando sensor L7...
  Array combinado (seca+humeda): shape = (8, 763, 542)
  Array escalado: shape = (8, 763, 542), NaN preservados = 23296
  Guardado: /content/drive/MyDrive/ANP/Landsat/SVerde_Landsat/SVD_tif-2000/Nympy2000/SVD_2000_L7_8capas_escalado.npy
  Guardado: /content/drive/MyDrive/ANP/Landsat/SVerde_Landsat/SVD_tif-2000/Nympy2000/SVD_2000_L7_8capas_orden.json

Orden de capas (igual para todos los sensores):
['NDVI_seca', 'TC_Brightness_seca', 'TC_Greenness_seca', 'TC_Wetness_seca', 'NDVI_humeda', 'TC_Brightness_humeda', 'TC_Greennes

## Verificación rápida
Estadísticas básicas del array escalado para confirmar que RobustScaler funcionó como se espera (mediana ≈ 0, IQR ≈ 1 por capa, ignorando NaN).

In [14]:
for sensor in SENSORES:
    arr = resultados[sensor]['array']
    orden = resultados[sensor]['orden_capas']
    print(f"\n{sensor}")
    print("-" * 40)

    for i in range(len(orden)):
        nombre = orden[i]
        capa = arr[i]
        valida = capa[~np.isnan(capa)]
        mediana = np.median(valida)
        q1 = np.percentile(valida, 25)
        q3 = np.percentile(valida, 75)
        print(f"{nombre}: mediana={mediana:.4f}  IQR={q3 - q1:.4f}  min={valida.min():.4f}  max={valida.max():.4f}")


L5
----------------------------------------
NDVI_seca: mediana=0.0000  IQR=1.0000  min=-5.2295  max=5.0626
TC_Brightness_seca: mediana=0.0000  IQR=1.0000  min=-5.2755  max=8.7345
TC_Greenness_seca: mediana=0.0000  IQR=1.0000  min=-3.5188  max=6.4409
TC_Wetness_seca: mediana=0.0000  IQR=1.0000  min=-7.6343  max=3.5390
NDVI_humeda: mediana=0.0000  IQR=1.0000  min=-6.5136  max=1.4444
TC_Brightness_humeda: mediana=0.0000  IQR=1.0000  min=-4.7539  max=14.1218
TC_Greenness_humeda: mediana=0.0000  IQR=1.0000  min=-3.0407  max=6.0211
TC_Wetness_humeda: mediana=0.0000  IQR=1.0000  min=-6.7413  max=3.3917

L7
----------------------------------------
NDVI_seca: mediana=0.0000  IQR=1.0000  min=-4.8592  max=4.2180
TC_Brightness_seca: mediana=0.0000  IQR=1.0000  min=-5.3574  max=10.0799
TC_Greenness_seca: mediana=0.0000  IQR=1.0000  min=-6.2243  max=4.3000
TC_Wetness_seca: mediana=0.0000  IQR=1.0000  min=-12.3086  max=3.4014
NDVI_humeda: mediana=0.0000  IQR=1.0000  min=-5.5001  max=2.1067
TC_Bright